Doing HVG selection in separate notebook so dependencies for scib don't get messed up

In [1]:
import os
import sys
import numpy as np
import scanpy as sc
from pathlib import Path

# version control
print("numpy:", np.__version__)
print("scanpy:", sc.__version__)

numpy: 1.26.4
scanpy: 1.10.2


In [2]:
def create_output_dir(master_dir, sub_dir_name, change_dir=False):
    '''
    Create an output directory as a subdirectory 'sub_dir_name' string within a parent directory master_dir
    Will not overwrite the files within that directory
    2025-05-28 moved to functions file
    2025-08-29 added change_dir
    '''
    output_dir = master_dir / sub_dir_name
    output_dir.mkdir(parents=True, exist_ok=True)
    print(f'Created output directory {output_dir}')
    if change_dir:
        os.chdir(output_dir)
        sc.settings.figdir = output_dir
        print(f'Default output directory changed to {output_dir}')
    return output_dir

In [3]:
# Set up input and output directories
CURRENT_DIR = Path.cwd()
PROJECT_DIR = CURRENT_DIR.parent
print(PROJECT_DIR)

DATA_DIR = PROJECT_DIR / 'objects'
print(DATA_DIR)

output_dir = create_output_dir(PROJECT_DIR, 'scib', change_dir=True)

/oak/stanford/groups/longaker/ULMS/revision/scRNAseq
/oak/stanford/groups/longaker/ULMS/revision/scRNAseq/objects
Created output directory /oak/stanford/groups/longaker/ULMS/revision/scRNAseq/scib
Default output directory changed to /oak/stanford/groups/longaker/ULMS/revision/scRNAseq/scib


In [4]:
adata = sc.read_h5ad(DATA_DIR / 'annotated_raw_counts.h5ad')
print(adata)

AnnData object with n_obs × n_vars = 223655 × 37964
    obs: 'batch', 'sample', 'n_counts', 'n_genes_by_counts', 'log1p_n_genes_by_counts', 'total_counts', 'log1p_total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'log1p_total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'log1p_total_counts_ribo', 'pct_counts_ribo', 'n_genes', 'doublet', 'doublet_score', 'celltype'


In [5]:
adata.layers["counts"] = adata.X.copy() # this layer will contain the raw counts
sc.pp.normalize_total(adata) # normalize X to the median total counts
sc.pp.log1p(adata) # logarithmize X
adata.raw = adata # full dimension normalized logtransformed raw data

In [6]:
# Calculate HVGs. We use 2000 per the Zappia et al feature selection paper
print(f"Number of genes before HVG selection: {adata.n_vars}")
sc.pp.highly_variable_genes(
    adata,
    flavor="seurat_v3",
    n_top_genes=2000,
    layer="counts",
    batch_key="batch",
    subset=True,
)
print(f"Number of genes after HVG selection: {adata.n_vars}")

Number of genes before HVG selection: 37964


/home/jpagolia/agolia_virtual_env/lib/python3.9/site-packages/numba/np/ufunc/parallel.py:371: NumbaWarning: The TBB threading layer requires TBB version 2021 update 6 or later i.e., TBB_INTERFACE_VERSION >= 12060. Found TBB_INTERFACE_VERSION = 12050. The TBB threading layer is disabled.
  warnings.warn(problem)


Number of genes after HVG selection: 2000


In [7]:
# Some cells may have zero HVG counts - this may mess up integration and differential expression calculation by creating a division by zero
print(f"Number of cells in anndata: {adata.n_obs}")
# Make sure to use the raw counts layer
low_counts = adata[adata.layers['counts'].sum(axis=1) < 1]
print(f"Number of cells with zero HVG counts: {low_counts.n_obs}")

Number of cells in anndata: 223655
Number of cells with zero HVG counts: 0


In [8]:
adata.write_h5ad('adata_hvg2000_beforescib.h5ad')